# Work@Home 4 - Text-to-SQL Chatbot

This project implements a chatbot that converts natural
language questions in Turkish or English into SQL queries.

It uses Groq LLM, JSON Schema, Python, and the Northwind
SQLite database.

## 1. Database Connection

Connect to the Northwind SQLite database in read-only mode.

In [26]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("file:northwind.db?mode=ro", uri=True)

tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()

print(tables)

[('Categories',), ('sqlite_sequence',), ('CustomerCustomerDemo',), ('CustomerDemographics',), ('Customers',), ('Employees',), ('EmployeeTerritories',), ('Order Details',), ('Orders',), ('Products',), ('Regions',), ('Shippers',), ('Suppliers',), ('Territories',)]


## 2. LLM Configuration

Initialize an OpenAI-compatible client using the Groq API.

In [27]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

## 3. Database Schema Extraction

Retrieve table definitions to help the LLM generate valid SQL.

In [28]:
schema = "\n".join(
    row[0] for row in conn.execute(
        """
        SELECT sql
        FROM sqlite_master
        WHERE type = 'table'
        AND name NOT LIKE 'sqlite_%'
        AND sql IS NOT NULL
        """
    )
)

print(schema[:1000])

CREATE TABLE [Categories]
(      [CategoryID] INTEGER PRIMARY KEY AUTOINCREMENT,
       [CategoryName] TEXT,
       [Description] TEXT,
       [Picture] BLOB
)
CREATE TABLE [CustomerCustomerDemo](
   [CustomerID]TEXT NOT NULL,
   [CustomerTypeID]TEXT NOT NULL,
   PRIMARY KEY ("CustomerID","CustomerTypeID"),
   FOREIGN KEY ([CustomerID]) REFERENCES [Customers] ([CustomerID]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION,
	FOREIGN KEY ([CustomerTypeID]) REFERENCES [CustomerDemographics] ([CustomerTypeID]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
)
CREATE TABLE [CustomerDemographics](
   [CustomerTypeID]TEXT NOT NULL,
   [CustomerDesc]TEXT,
    PRIMARY KEY ("CustomerTypeID")
)
CREATE TABLE [Customers]
(      [CustomerID] TEXT,
       [CompanyName] TEXT,
       [ContactName] TEXT,
       [ContactTitle] TEXT,
       [Address] TEXT,
       [City] TEXT,
       [Region] TEXT,
       [PostalCode] TEXT,
       [Country] TEXT,
       [Phone] TEXT,
       [Fax] TEXT,
  


## 4. System Prompt and Structured Output

Define the SQL generation rules and enforce a JSON Schema
with answerable, sql, and explanation fields.

In [29]:
SYSTEM_PROMPT = f"""
You are a SQL assistant for a company manager.

The user can ask questions in Turkish or English.

Generate exactly one SQLite SELECT query
using only the provided database schema.

Rules:
- Only generate SELECT queries.
- Do not generate INSERT, UPDATE, DELETE or DROP queries.
- If the question cannot be answered, set answerable to false.
- If answerable is false, return an empty SQL string.
- Write explanations in the same language as the user.

Database schema:
{schema}
"""

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "sql_answer",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "answerable": {"type": "boolean"},
                "sql": {"type": "string"},
                "explanation": {"type": "string"}
            },
            "required": ["answerable", "sql", "explanation"],
            "additionalProperties": False
        }
    }
}

## 5. SQL Generation and Execution

Generate SQL using the LLM, validate the query,
execute it on SQLite, and display the results.

In [30]:
import json
def ask(question):

    # 1. Ask the LLM to generate SQL
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        response_format=RESPONSE_FORMAT
    )

    # 2. Convert JSON string to Python dictionary
    data = json.loads(response.choices[0].message.content)

    print("\nJSON Response:")
    print(json.dumps(data, indent=2, ensure_ascii=False))

    # 3. Check whether the question is answerable
    if not data["answerable"]:
        print(data["explanation"])
        return

    # 4. Validate SQL
    sql = data["sql"].strip()

    if not sql.lower().startswith("select"):
        print("Only SELECT queries are allowed.")
        return

    sql = sql.rstrip(";").strip()

    if ";" in sql:
        print("Multiple SQL statements are not allowed.")
        return

    print("\nGenerated SQL:")
    print(sql)

    # 5. Execute SQL
    try:
        conn.execute("PRAGMA query_only = ON")
        df = pd.read_sql_query(sql, conn)

        print("\nDatabase Result:")
        display(df.head(20))

        print("\nExplanation:", data["explanation"])

    except Exception as e:
        print("SQL Error:", e)

## 6. Interactive Chatbot

Allow the user to ask multiple questions until
they enter "exit".

In [31]:
test_questions = [
    "Hangi ürün kategorisinde kaç ürün bulunuyor?",
    "Toplam satış tutarı en yüksek 3 ürünü göster.",
    "List the products that are currently out of stock.",
    "Delete all customers from the database.",
    "2027 yılı için şirketin tahmini kârı ne kadar?"
]

for question in test_questions:
    print("\n" + "=" * 60)
    print("QUESTION:", question)

    try:
        ask(question)
    except Exception as e:
        print("Error:", e)


QUESTION: Hangi ürün kategorisinde kaç ürün bulunuyor?

JSON Response:
{
  "answerable": true,
  "explanation": "Aşağıdaki sorgu, her bir ürün kategorisindeki ürün sayısını döndürür.",
  "sql": "SELECT c.CategoryName, COUNT(p.ProductID) AS ProductCount\nFROM Products p\nJOIN Categories c ON p.CategoryID = c.CategoryID\nGROUP BY c.CategoryID, c.CategoryName;"
}

Generated SQL:
SELECT c.CategoryName, COUNT(p.ProductID) AS ProductCount
FROM Products p
JOIN Categories c ON p.CategoryID = c.CategoryID
GROUP BY c.CategoryID, c.CategoryName

Database Result:


,CategoryName,ProductCount
0,Beverages,12
1,Condiments,12
2,Confections,13
3,Dairy Products,10
4,Grains/Cereals,7
5,Meat/Poultry,6
6,Produce,5
7,Seafood,12



Explanation: Aşağıdaki sorgu, her bir ürün kategorisindeki ürün sayısını döndürür.

QUESTION: Toplam satış tutarı en yüksek 3 ürünü göster.

JSON Response:
{
  "answerable": true,
  "explanation": "Toplam satış tutarı, ürün başına toplam gelir (UnitPrice × Quantity × (1 – Discount)) hesaplanarak belirlenir. Ürünlere göre toplama yapılır, en yüksek değere sahip üç ürün sıralanır ve limit 3 ile sınırlandırılır.",
  "sql": "SELECT p.ProductID, p.ProductName, SUM(d.UnitPrice * d.Quantity * (1 - d.Discount)) AS TotalSales\nFROM `Order Details` d\nJOIN Products p ON d.ProductID = p.ProductID\nGROUP BY p.ProductID, p.ProductName\nORDER BY TotalSales DESC\nLIMIT 3;"
}

Generated SQL:
SELECT p.ProductID, p.ProductName, SUM(d.UnitPrice * d.Quantity * (1 - d.Discount)) AS TotalSales
FROM `Order Details` d
JOIN Products p ON d.ProductID = p.ProductID
GROUP BY p.ProductID, p.ProductName
ORDER BY TotalSales DESC
LIMIT 3

Database Result:


,ProductID,ProductName,TotalSales
0,38,Côte de Blaye,141396.735
1,29,Thüringer Rostbratwurst,80368.672
2,59,Raclette Courdavault,71155.700



Explanation: Toplam satış tutarı, ürün başına toplam gelir (UnitPrice × Quantity × (1 – Discount)) hesaplanarak belirlenir. Ürünlere göre toplama yapılır, en yüksek değere sahip üç ürün sıralanır ve limit 3 ile sınırlandırılır.

QUESTION: List the products that are currently out of stock.

JSON Response:
{
  "answerable": true,
  "explanation": "Stokta bulunmayan ürünleri bulmak için Ürünlerin UnitsInStock alanı sıfır olan kayıtları sorguluyoruz.",
  "sql": "SELECT * FROM Products WHERE UnitsInStock = 0;"
}

Generated SQL:
SELECT * FROM Products WHERE UnitsInStock = 0

Database Result:


,ProductID,ProductName,SupplierID,CategoryID,QuantityPerUnit,UnitPrice,UnitsInStock,UnitsOnOrder,ReorderLevel,Discontinued
0,5,Chef Anton's Gumbo Mix,2,2,36 boxes,21.35,0,0,0,1
1,17,Alice Mutton,7,6,20 - 1 kg tins,39.00,0,0,0,1
2,29,Thüringer Rostbratwurst,12,6,50 bags x 30 sausgs.,123.79,0,0,0,1
3,31,Gorgonzola Telino,14,4,12 - 100 g pkgs,12.50,0,70,20,0
4,53,Perth Pasties,24,6,48 pieces,32.80,0,0,0,1



Explanation: Stokta bulunmayan ürünleri bulmak için Ürünlerin UnitsInStock alanı sıfır olan kayıtları sorguluyoruz.

QUESTION: Delete all customers from the database.

JSON Response:
{
  "answerable": false,
  "explanation": "I can only generate SELECT queries, and the request is to delete all customers, which cannot be answered with a SELECT statement.",
  "sql": ""
}
I can only generate SELECT queries, and the request is to delete all customers, which cannot be answered with a SELECT statement.

QUESTION: 2027 yılı için şirketin tahmini kârı ne kadar?

JSON Response:
{
  "answerable": false,
  "explanation": "Veritabanında 2027 yılı kârını hesaplamak için gerekli bilgiler yok.",
  "sql": ""
}
Veritabanında 2027 yılı kârını hesaplamak için gerekli bilgiler yok.


In [32]:
while True:
    question = input("You (exit to quit): ")

    if question.strip().lower() == "exit":
        print("Goodbye!")
        break

    if not question.strip():
        continue

    try:
        ask(question)
    except Exception as e:
        print("Error:", e)


JSON Response:
{
  "answerable": false,
  "explanation": "Anlamlı bir soru belirtmediğiniz için cevap veremiyorum.",
  "sql": ""
}
Anlamlı bir soru belirtmediğiniz için cevap veremiyorum.

JSON Response:
{
  "answerable": false,
  "explanation": "Soru net değil. Lütfen daha fazla ayrıntı verin.",
  "sql": ""
}
Soru net değil. Lütfen daha fazla ayrıntı verin.
Goodbye!
